# <b>ПЛАН Б v3: дообучение Айки в Google Colab</b>
Qwen2.5-1.5B-Instruct + QLoRA на 40к ваших диалогах.

<b>Порядок:</b> Среда выполнения → Сменить среду → T4 GPU → «Выполнить всё».
Ячейка с Google Диском попросит доступ — разреши.
Каждый шаг печатает проверку. Чекпоинты льются на Диск каждые ~200 шагов, так что даже если сессия отвалится — ничего не сгорит.

In [ ]:
# 1. Проверка GPU. Должно быть Tesla T4 и ~15 ГБ
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'GPU НЕ ВКЛЮЧЁН! Среда выполнения → Сменить среду → T4 GPU'
print('GPU ок:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Ставим зависимости (фиксированные версии, без конфликта с JAX). ~2-3 мин
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
!pip uninstall -q -y jax jaxlib flax 2>/dev/null
!pip install -q transformers==4.46.3 tokenizers==0.20.3 huggingface_hub==0.26.2 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.1.1
import transformers, peft, bitsandbytes
print('версии:', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

In [ ]:
# 3. Подключаем Google Диск (появится ссылка → войди → разреши)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/aika-checkpoints', exist_ok=True)
print('Диск подключён')

In [ ]:
# 4. Забираем ваши 40к диалогов из репы aika-finetune
!wget -q https://github.com/lucifermornngstar52-cell/aika-finetune/raw/main/data/train.jsonl -O train.jsonl
!wget -q https://github.com/lucifermornngstar52-cell/aika-finetune/raw/main/data/val.jsonl -O val.jsonl
import json
lines = open('train.jsonl', encoding='utf-8').read().strip().split('\n')
data = [json.loads(l)['messages'] for l in lines]
print(f'диалогов: {len(data):,} | пример: {data[0][0]["content"][:60]}...')

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
# 5. Загружаем мозг Qwen в 4 битах (~1 ГБ, минуты 3)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={'': 0})
model.config.use_cache = False
print('влезло в VRAM: %.1f ГБ' % (torch.cuda.memory_allocated()/1e9))

In [ ]:
# 6. LoRA-слой — тот самый «1% души»
from peft import LoraConfig, get_peft_model
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
                  task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 7. Готовим данные в формат чата Qwen
import torch
class DialogDataset(torch.utils.data.Dataset):
    def __init__(self, msgs, tok, max_len=512):
        self.items = []
        for m in msgs:
            ids = tok.apply_chat_template(m, add_generation_prompt=False, return_tensors='pt')
            self.items.append(ids[0][:max_len])
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return {'input_ids': self.items[i]}

train_ds = DialogDataset(data, tok)
print(f'готово: {len(train_ds):,} примеров')

In [ ]:
# 8. ОБУЧЕНИЕ (~30-40 мин на T4). Чекпоинт на Диск каждые 200 шагов.
from transformers import TrainingArguments, Trainer
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def collate(features):
    maxlen = max(len(f['input_ids']) for f in features)
    ids, labels, att = [], [], []
    for f in features:
        x = f['input_ids'].tolist()
        n = maxlen - len(x)
        ids.append(x + [PAD]*n)
        labels.append(x + [-100]*n)
        att.append([1]*len(x) + [0]*n)
    return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels), 'attention_mask': torch.tensor(att)}

args = TrainingArguments(
    output_dir='/content/drive/MyDrive/aika-checkpoints',
    num_train_epochs=1, per_device_train_batch_size=4,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
    logging_steps=25, save_strategy='steps', save_steps=200, save_total_limit=1,
    save_only_model=True, fp16=True, optim='paged_adamw_8bit',
    report_to='none', lr_scheduler_type='cosine', seed=42)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
trainer.train()
print('ГОТОВО. Последний loss:', trainer.state.log_history[-1].get('train_loss', '?'))

In [ ]:
# 9. Проверяем, как она теперь говорит
model.config.use_cache = True
model.eval()
def ask(q):
    ids = tok.apply_chat_template([{'role':'user','content':q}], add_generation_prompt=True,
                                   return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=120, temperature=0.7, top_p=0.9,
                         do_sample=True, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
for q in ['привет, как дела?', 'что делать если скучно?', 'расскажи о себе']:
    print(f'ТЫ: {q}\nАЙКА: {ask(q)}\n')

In [ ]:
# 10. Сливаем LoRA в один мозг и сохраняем на Диск (~3 ГБ, минуты 5)
merged = model.merge_and_unload()
merged = merged.half()
save = '/content/drive/MyDrive/aika-qwen-1.5b'
merged.save_pretrained(save)
tok.save_pretrained(save)
print('сохранено:', save)

In [ ]:
# 11. (опционально) GGUF для телефона / llama.cpp. ~5 мин, БЕЗ компиляции.
!pip install -q gguf sentencepiece
!git clone -q --depth 1 https://github.com/ggerganov/llama.cpp /content/llamac
!python /content/llamac/convert_hf_to_gguf.py /content/drive/MyDrive/aika-qwen-1.5b --outfile /content/drive/MyDrive/aika-q8_0.gguf --outtype q8_0
import os
print('GGUF готов: %.1f ГБ' % (os.path.getsize('/content/drive/MyDrive/aika-q8_0.gguf')/1e9))